<a href="https://colab.research.google.com/github/Jupeid/interactivebook/blob/main/TesteLivroInt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
class Personagem:
  def __init__(self,nome):
    self.nome = nome
    self.nivel = 0
    self.experiencia = 0
    self.exp_para_proximo_nivel = 10
    self.pontos_disponiveis = 0
    self.pode_distribuir_pontos = False
# ------ATRIBUTOS BASE------
    self.forca = 0
    self.destreza = 0
    self.inteligencia = 1
    self.sorte = 1
    self.magia = 0
    self.vitalidade = 0
# ------STATUS DERIVADOS------
    self.hp_max = 20 + (self.vitalidade * 5)
    self.hp_atual = self.hp_max
    self.mp_max = self.magia * 5
    self.mp_atual = self.mp_max
# -----COMPANIONS/INIMIGOS------
    self.npcs = {}
# -----INVENTARIO------
    self.inventario = []
# -----METODOS DE NPCS------
  def adicionar_npc(self, id_npc, nome, faccao="neutro"):
    if id_npc not in self.npcs:
      self.npcs[id_npc] = NPC(nome=nome, faccao=faccao)
      print(f"\n [MUNDO] Você conheceu: {nome}!")

  def obter_npc(self, id_npc):
    return self.npcs.get(id_npc)

# -----METODOS DE INVENTARIO------
  def adicionar_item(self, item):
    if item not in self.inventario:
      self.inventario.append(item)
      print(f"\n [INVENTARIO] Você adquiriu: '{item}'!")

  def remover_item(self, item):
    if item in self.inventario:
      self.inventario.remove(item)
      print(f"\n [INVENTARIO] Você perdeu: '{item}'!")
      return True
    return False
  def tem_item(self, item):
    return item in self.inventario
# ------METODO DE TRAVA----
  def permitir_distribuicao(self, permitir:bool):
    self.pode_distribuir_pontos = permitir
# ------ATT DE STATUS------
  def recalcular_status_maximos(self):
    self.hp_max = 20 + (self.vitalidade * 5)
    self.mp_max = self.magia * 5
# ------TESTE ATRIBUTOS-----
  def tem_atributo(self, nome_atributo, valor_minimo):
    valor_atual = getattr(self, nome_atributo.lower(), 0)
    return valor_atual >= valor_minimo

  def tem_mp(self, custo_mp):
    return self.mp_atual >= custo_mp
# -----ALTERA ATRIBUTOS-----
  def consumir_mp(self, quantidade):
    if self.tem_mp(quantidade):
      self.mp_atual -= quantidade
      print(f"\n [MP] Você gastou {quantidade} de MP. MP Atual: {self.mp_atual}/{self.mp_max}")
      return True
    return False

  def receber_dano(self, dano):
    self.hp_atual = max(0, self.hp_atual - dano)
    print(f"\n [DANO] Você sofreu {dano} de dano! HP: {self.hp_atual}/{self.hp_max}")

  def descansar(self):
    self.hp_atual = self.hp_max
    self.mp_atual = self.mp_max
    print("\n [DESCANSO] Seu HP e MP forma totalmente restaurados!")

# -----SISTEMA DE NIVEL E EXPERIENCIA-----
  def ganhar_xp(self, quantidade_xp):
    self.experiencia += quantidade_xp
    print(f"\n [XP] Você ganhou {quantidade_xp} de XP! ({self.experiencia}/{self.exp_para_proximo_nivel})")

    while self.experiencia >= self.exp_para_proximo_nivel:
        self.experiencia -= self.exp_para_proximo_nivel
        self.nivel += 1
        self.pontos_disponiveis += 1
        self.exp_para_proximo_nivel = int(self.exp_para_proximo_nivel * 1.5)
        print(f"\n NIVEL UP! Você subiu para o Nível {self.nivel}!")
        print(f"Você tem {self.pontos_disponiveis} ponto(s) de status para distribuir.")

  def distribuir_pontos(self, atributo):
    if not self.pode_distribuir_pontos:
      print("\n [!] Você só pode distribuir pontos em áreas de descanso"
      " ou trocas de capítulo!")
      return False

    if self.pontos_disponiveis <= 0:
      print("Você não possui pontos de status disponíveis!")
      return False

    atributo = atributo.lower()
    atributos_validos = [
        "forca",
        "destreza",
        "inteligencia",
        "sorte",
        "magia",
        "vitalidade",
    ]
    if atributo in atributos_validos and hasattr(self, atributo):
      valor_antigo = getattr(self, atributo)
      setattr(self, atributo, valor_antigo + 1)
      self.pontos_disponiveis -= 1

      self.recalcular_status_maximos()

      if atributo == "vitalidade":
        self.hp_atual += 5
      elif atributo == "magia":
        self.mp_atual += 5

      print(f"\n [STATUS] {atributo.capitalize()} aumentado para {getattr(self, atributo)}!")
      return True
    else:
      print(f"[!] Atributo inválido!")
      return False

In [33]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
class NPC:
    def __init__(self, nome, faccao="neutro"):
        self.nome = nome
        self.faccao = faccao #aliado, inimigo, neutro
        self.vivo = True
        self.imobilizado = False

    def esta_disponivel(self):
        return self.vivo and not self.imobilizado

    def abater(self):
        self.vivo = False
        print(f"\n [MUNDO] {self.nome} foi abatido!")

    def imobilizar(self):
        self.imobilizado = True
        print(f"\n [MUNDO] {self.nome} está imobilizado!")

In [35]:
def testar_requisitos(jogador, lista_requisitos):
  for req in lista_requisitos:
    tipo = req["tipo"]

    if tipo == "atributo":
      if not jogador.tem_atributo(req["nome"], req["valor"]):
        return False
    elif tipo == "magia":
      tem_nivel = jogador.tem_atributo(req["nome"], req["valor"])
      tem_mp = jogador.tem_mp(req["custo_mp"])
      if not (tem_nivel and tem_mp):
        return False
    elif tipo == "item":
      # Changed from `[i.nome for i in jogador.inventario]` to `jogador.inventario`
      if req["nome"] not in jogador.inventario:
        return False
  return True

In [36]:
def processar_escolha(jogador, opcao_escolhida):
  if "proxima_cena" in opcao_escolhida:
    return opcao_escolhida["proxima_cena"]

  modos = opcao_escolhida.get("modos", [])
  modo_bem_sucedido = None

  for modo in modos:
    if testar_requisitos(jogador, modo["requisitos"]):
      modo_bem_sucedido = modo
      break
  if modo_bem_sucedido:
    for req in modo_bem_sucedido["requisitos"]:
      if req["tipo"] == "magia":
        jogador.consumir_mp(req["custo_mp"])

    narrativa_formatada = formatar_texto(modo_bem_sucedido["narrativa"])

    print("\n" + "=" * 40)
    print(narrativa_formatada)
    print("=" * 40 + "\n")

    if "dano_recebido" in modo_bem_sucedido:
      dano = modo_bem_sucedido["dano_recebido"]
      jogador.receber_dano(dano)

    if "item_removido" in modo_bem_sucedido:
      for item in modo_bem_sucedido["item_removido"]:
        jogador.remover_item(item)

    if "item_adquirido" in modo_bem_sucedido:
      for item in modo_bem_sucedido["item_adquirido"]:
        jogador.adicionar_item(item)

    if "xp_ganha" in modo_bem_sucedido:
      jogador.ganhar_xp(modo_bem_sucedido["xp_ganha"])

    if jogador.hp_atual <=0:
      print("\n[GAME OVER] Você sucumbiu aos ferimentos...")
      return None

    if "novo_npc" in modo_bem_sucedido:
      dados = modo_bem_sucedido["novo_npc"]
      jogador.adicionar_npc(
        id_npc=dados["id_npc"],
        nome=dados["nome"],
        faccao=dados.get("faccao", "neutro")
      )

    if "efeito_npc" in modo_bem_sucedido:
      efeito = modo_bem_sucedido["efeito_npc"]
      npc = jogador.obter_npc(efeito["id_npc"])
      if npc:
        if efeito["acao"] == "imobilizar":
          npc.imobilizar()
        elif efeito["acao"] == "abater":
          npc.abater()
        elif efeito["acao"] == "mudar_faccao":
          npc.faccao = efeito["nova_faccao"]

    return modo_bem_sucedido["proxima_cena"]

  else:
    print("\n[!] Escolha inválida!" )
    return None

In [37]:
def exibir_hud(jogador):
  print("=" * 50)
  print(f"Nome: {jogador.nome} | Nível: {jogador.nivel} | Pontos Disponíveis: {jogador.pontos_disponiveis}")
  print(f"HP: {jogador.hp_atual}/{jogador.hp_max} | MP: {jogador.mp_atual}/{jogador.mp_max}")
  print(f"Força: {jogador.forca} | Destreza: {jogador.destreza}")
  print(f"Inteligência: {jogador.inteligencia} | Sorte: {jogador.sorte}")
  print(f"Magia: {jogador.magia} | Vitalidade: {jogador.vitalidade}")

  if jogador.inventario:
    itens_str = ", ".join(jogador.inventario)
  else:
    itens_str = "Vazio"
  print(f"Inventário/Bençãos: {itens_str}")
  print("=" * 50)

In [38]:
def avaliar_condicao_oculta(jogador, condicao):
    tipo = condicao.get("tipo")

    if tipo == "atributo":
        return jogador.tem_atributo(condicao["nome"], condicao["valor"])

    elif tipo == "item":
        return jogador.tem_item(condicao["nome"])

    elif tipo == "hp_minimo":
        return jogador.hp_atual >= condicao["valor"]

    elif tipo == "mp_minimo":
        return jogador.mp_atual >= condicao["valor"]

    return False

In [39]:
import textwrap
import os

def obter_diretorio_base():
  caminho_drive = "/content/drive/MyDrive/Colab Notebooks"

  if os.path.exists("/content/drive"):
    return caminho_drive

  if "__file__" in globals():
    return os.path.dirname(os.path.abspath(__file__))
  return os.getcwd()

def carregar_texto(caminho_arquivo, largura=80):
    diretorio_atual = obter_diretorio_base()
    caminho_completo = os.path.join(diretorio_atual, caminho_arquivo)

    try:
        with open(caminho_completo, 'r', encoding='utf-8') as arquivo:
            conteudo = arquivo.read()
            return formatar_texto(conteudo, largura=largura)

    except FileNotFoundError:
        print(f"[!] Arquivo não encontrado: {caminho_completo}")
        return "Texto indisponível no momento."

def formatar_texto(texto, largura=80):
    if not texto:
        return ""

    paragrafos = texto.split('\n\n')
    paragrafos_formatados = []

    for p in paragrafos:
        p_limpo = " ".join(p.split())
        p_formatado = textwrap.fill(p_limpo, width=largura)
        paragrafos_formatados.append(p_formatado)

    return "\n\n".join(paragrafos_formatados)

In [40]:
def rodar_cena(jogador, cena):
  if "checks_ocultos" in cena:
    for condicao in cena["checks_ocultos"]:
     if avaliar_condicao_oculta(jogador, condicao):
       return condicao["cena_sucesso"]
    return cena["cena_falha"]

  permite_descanso = cena.get("permite_descanso", False)
  jogador.permitir_distribuicao(permite_descanso)
  if permite_descanso:
    jogador.descansar()

  while True:
    exibir_hud(jogador)

    titulo = cena.get("titulo")

    if titulo:
      print("\n" + "=" * 40)
      print(cena["titulo"])
      print("=" * 40)
    else:
      print("\n")

    print("\n" + formatar_texto(cena["narrativa"]))

    if "proxima_cena" in cena and "opcoes" not in cena:
      input("\nPressione Enter para continuar...")
      return cena["proxima_cena"]

    if "opcoes" not in cena or not cena["opcoes"]:
      print("\n[Fim deste capítulo]")
      return None

    print("\nOpções:")
    for letra, dados_opcao in cena["opcoes"].items():
      texto_opcao = formatar_texto(dados_opcao['texto'])
      print(f"[{letra}] {texto_opcao}")

    entrada = input("\nEscolha uma opção: ").strip().upper()
    if entrada not in cena["opcoes"]:
      print("\n[!] Escolha inválida!")
      input("Pressione Enter para tentar novamente...")
      continue

    opcoes_selecionadas = cena["opcoes"][entrada]
    proxima_cena = processar_escolha(jogador, opcoes_selecionadas)

    if proxima_cena is not None:
      return proxima_cena

    input("Pressione Enter para tentar novamente...")

In [41]:
cenas = {
    "prologo": {
        "titulo": "PRÓLOGO - O JOGO DIVINO",
        "narrativa": carregar_texto("prologo.txt"),
        "proxima_cena": "capitulo_1",
    },

    "capitulo_1": {
      "titulo": "Capítulo 1 – O Primeiro Encontro",
      "narrativa": carregar_texto("capitulo_1.txt"),
      "opcoes": {
            "A": {
                "texto": """Tentar atrair a atenção dos lobos, criando uma brecha para que a garota finalize a conjuração do feitiço. (Inteligência 1)""",
                "modos": [
                    {
                      "requisitos": [
                            {
                                "tipo": "atributo",
                                "nome": "inteligencia",
                                "valor":1
                            }
                        ],
                      "narrativa": """Noto que, próximo aos incensos, há algumas flores que, quando queimadas junto ao odor da erva, geram o efeito oposto: em vez de atrair, afastam os predadores. Chuto os incensos com força na direção do arbusto florido. Os lobos se assustam com o movimento repentino e recuam alguns passos.""",
                      "proxima_cena": "capitulo_1_resgate",
                      "xp_ganha": 5,
                    }
                ]
            },
          "B": {
              "texto": """Colocar-me entre os lobos e a garota, destruindo os incensos na esperança de dispersar a agressividade dos predadores.""",
              "modos": [
                  {
                      "requisitos": [],
                      "narrativa": """Corro na direção da garota, colocando-me entre os lobos e sua presa. Um deles salta no mesmo instante, fazendo com que eu caia ao lado de um dos incensos. Aproveitando o momento de impacto no chão, arremesso o incenso na direção dos animais. Ele cai sobre um arbusto florido, que começa a queimar e exalar uma fumaça densa, deixando os lobos desorientados por um breve momento.""",
                      "dano_recebido": 5,
                      "xp_ganha": 5,
                      "proxima_cena": "capitulo_1_resgate",
                  }
              ]
            },
#          "C": {
#              "texto": """Ignorar o apelo da garota e a notificação do Sistema, virando as costas e indo embora.""",
#              "modos": [
#                  {
#                      "requisitos": [],
#                      "narrativa": """Decido que não vale a pena arriscar minha vida por uma desconhecida. Viro as costas e me preparo para ir embora.""",
#                      "proxima_cena": "capitulo_1_abandono",
#                  }
#              ],
#            },
        }
    },

    "capitulo_1_resgate": {
        "narrativa": carregar_texto("capitulo_1_resgate.txt"),
        "opcoes": {
            "A": {
                "texto": "Apresentar-me e dizer meu nome.",
                "modos": [
                    {
                        "requisitos": [],
                        "narrativa": """— Meu nome é Kael — respondo, ajudando-a a se sentar. — Sou morador de um vilarejo próximo e notei a movimentação incomum na mata. Ainda bem que consegui chegar a tempo.""",
                        "novo_npc": {
                            "id_npc": "campeao_astrid",
                            "nome": "Astrid",
                            "faccao": "aliado"
                        },
                        "proxima_cena": "capitulo_1_lobo",

                    }
                ]
            },
            "B": {
                "texto": "Ignorar a pergunta e focar em sair do local rapidamente.",
                "modos": [
                    {
                        "requisitos": [],
                        "narrativa": """— Meu nome não é importante no momento — digo de forma calma. — Vamos cuidar das suas feridas antes de pensarmos em retornar ao vilarejo.""",
                        "novo_npc": {
                            "id_npc": "campeao_astrid",
                            "nome": "Astrid",
                            "faccao": "aliado"
                        },
                        "proxima_cena": "capitulo_1_lobo",
                    }
                ]
            },
        }
    },

    "capitulo_1_lobo": {
        "narrativa": carregar_texto("capitulo_1_lobo.txt"),
        "opcoes": {
            "A": {
                "texto": "Pedir para que Astrid finalize o lobo e complete a missão extra.",
                "modos":[
                    {
                        "requisitos": [],
                        "narrativa": """Astrid reúne o restante das forças que lhe sobram e dispara mais uma lâmina de vento, cortando a garganta do lobo ferido.
===========================================================
[Missão Concluída!]
Aura está satisfeita, garantindo ao hospedeiro uma
oportunidade de evolução. Escolha com sabedoria!

[Nota Extra de Nox]
Você completou a missão extra. Com isso, tem o direito de
salvar uma vida para manter o equilíbrio sobre a vida perdida.
Use com sabedoria.
===========================================================
RECOMPENSAS:
[+5 XP]
[Pílula de Restauração]
[Bênção de Nox] (Permite sentir quando alguém está próximo
de se encontrar com o Deus da Morte)
=========================================================== """,
                    "item_adquirido": ["Pílula de Restauração", "Bênção de Nox"],
                    "xp_ganha": 5,
                    "proxima_cena": "capitulo_1_casa",
                    }
                ]
            },
            "B": {
                "texto": "Deixar o lobo escapar para preservar a mana da garota até sairmos da floresta.",
                "modos": [
                    {
                        "requisitos": [],
                        "narrativa": """Decido poupar as energias de Astrid. Deixo o animal assustado fugir manco pela vegetação.
===========================================================
[Missão Concluída!]
Aura está satisfeita, garantindo ao hospedeiro uma
oportunidade de evolução. Escolha com sabedoria!
===========================================================
RECOMPENSAS:
[+5 XP]
=========================================================== """,
                        "xp_ganha": 5,
                        "proxima_cena": "capitulo_1_casa",
                    }
                ]
            },
        }
    },

    "capitulo_1_casa": {
        "narrativa": carregar_texto("capitulo_1_casa.txt"),
        "proxima_cena": "cap_1_check_reacao_eskil"
    },

    "cap_1_check_reacao_eskil": {
        "checks_ocultos": [
            {
                "tipo": "hp_minimo",
                "valor": 20,
                "cena_sucesso": "capitulo_1_casa_A"
            }
        ],
        "cena_falha": "capitulo_1_casa_B",
    },

    "capitulo_1_casa_A": {
        "narrativa": """Eskil mostra aos presentes os restos das ervas queimadas, identificando o "repelente" improvisado que criei na batalha. Olho impressionado para o meu pai; ele conseguiu deduzir minha estratégia observando apenas os poucos rastros do combate.""",
        "proxima_cena": "capitulo_1_casa_final",
    },

    "capitulo_1_casa_B": {
        "narrativa": """Eskil mostra as ervas queimadas que formaram o repelente. Um suor frio escorre pela minha testa. Mal sabem eles que tudo não passou de uma grata e desesperada coincidência...""",
        "proxima_cena": "capitulo_1_casa_final",
    },

    "capitulo_1_casa_final": {
        "narrativa": carregar_texto("capitulo_1_casa_final.txt"),
        "checks_ocultos": [
            {
                "tipo": "item",
                "nome": "Bênção de Nox",
                "cena_sucesso": "capitulo_1_casa_final_bencao",
            }
        ],
        "cena_falha": "capitulo_1_casa_final_sem_bencao",
    },

    "capitulo_1_casa_final_bencao": {
        "narrativa": """Enquanto observo Iris puxar Astrid pela mão em direção ao meu quarto, noto uma névoa escura e opaca flutuando ao redor da minha irmã.
A interface translúcida do Sistema surge imediatamente diante dos meus olhos:

===========================================================
[Aviso de Nox]
• Você pode utilizar a [Pílula de Restauração] para curar Iris.
• Instruções: Misture o medicamento na sopa dela durante o
  jantar. A condição crônica será curada após uma noite de sono.
=========================================================== """,
        "opcoes": {
            "A": {
                "texto": "Usar a Pílula na sopa de Iris (-1 Pílula de Restauração)",
                "modos": [
                    {
                        "requisitos": [
                            {
                                "tipo": "item",
                                "nome": "Pílula de Restauração"
                            }
                        ],
                        "narrativa": """Durante o jantar, misturo a Pílula de Restauração na sopa de Iris. Ela come sem perceber, é possível notar a névoa escura ao redor dela se dissipando lentamente.""",
                        "item_removido": ["Pílula de Restauração"],
                        "proxima_cena": "capitulo_1_casa_jantar_bencao",
                    }
                ],
            },

            "B": {
                "texto": "Guardar a Pílula e não fazer nada",
                "modos": [
                    {
                        "requisitos": [],
                        "proxima_cena": "capitulo_1_casa_final_sem_bencao",
                    }
                ],
            },
        },
    },

    "capitulo_1_casa_final_sem_bencao": {
        "narrativa": "[CAPITULO EM CONSTRUÇÃO]",
        "proxima_cena": "caminho_neutro"
    },

    "capitulo_1_casa_jantar_bencao": {
        "narrativa": "[CAPITULO EM CONSTRUÇÃO]",
        "proxima_cena": "caminho_heroico"
    },
}



In [42]:
protagonista = Personagem("Kael")
cena_atual_id = "prologo"

while cena_atual_id in cenas:
  cena_objeto = cenas[cena_atual_id]
  cena_atual_id = rodar_cena(protagonista, cena_objeto)
print("\n[Fim do Livro Interativo]")

Nome: Kael | Nível: 0 | Pontos Disponíveis: 0
HP: 20/20 | MP: 0/0
Força: 0 | Destreza: 0
Inteligência: 1 | Sorte: 1
Magia: 0 | Vitalidade: 0
Inventário/Bençãos: Vazio

PRÓLOGO - O JOGO DIVINO

Em um salão celestial banhado por uma luz dourada, oito entidades se reúnem ao
redor de uma imensa mesa circular ornada em ouro. Ao fundo da sala, sobre um
pedestal elevado, repousa um trono majestoso, porém vazio. Em ambos os lados
desse assento principal, dois tronos menores completam a arquitetura divina. No
trono da direita, descansa uma figura de pele azulada, envolta em robes negros
trespassados por linhas douradas que se movem como areia viva. O capuz cobre-lhe
a cabeça e sua feição permanece imutável, enquanto seus olhos azuis brilhantes
observam em silêncio o debate dos deuses abaixo. Ao seu lado, no trono da
esquerda, dividem o espaço duas figuras: um homem adulto, aparentando trinta
anos, vestindo robes negros encapuzados e portando olhos e cabelos tão escuros
quanto o vazio; em seu co

KeyboardInterrupt: Interrupted by user